# Molecular Property Prediction Demo — molprop-gnn

This notebook demonstrates the GIN-based molecular property prediction pipeline:
1. Input a SMILES string
2. Featurize the molecule into a graph
3. Run inference through the GIN model
4. Show prediction with uncertainty estimate
5. Visualize atom-level attention weights

We use aspirin (acetylsalicylic acid) as our example molecule.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

print(f"torch: {torch.__version__}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"device: {device}")

# property targets (MoleculeNet ESOL, FreeSolv, Lipophilicity benchmarks)
PROPERTY_NAMES = ['logS (ESOL)', 'logP (Lipophilicity)', 'ΔG_hyd (FreeSolv)']

## Input SMILES and Featurization

We canonicalize the SMILES string first to ensure consistent atom ordering,
then convert to a molecular graph with atom and bond features.

In [ ]:
from src.data.featurizer import MoleculeFeaturizer

# example molecules
molecules = {
    'aspirin':     'CC(=O)Oc1ccccc1C(=O)O',
    'caffeine':    'Cn1cnc2c1c(=O)n(c(=O)n2C)C',
    'ibuprofen':   'CC(C)Cc1ccc(cc1)C(C)C(=O)O',
    'paracetamol': 'CC(=O)Nc1ccc(O)cc1',
}

smiles = molecules['aspirin']
print(f"input SMILES: {smiles}")

featurizer = MoleculeFeaturizer()

try:
    from rdkit import Chem
    mol = Chem.MolFromSmiles(smiles)
    canonical = Chem.MolToSmiles(mol)
    print(f"canonical SMILES: {canonical}")
    print(f"num atoms: {mol.GetNumAtoms()}")
    print(f"num bonds: {mol.GetNumBonds()}")
except ImportError:
    print("rdkit not installed — skipping canonicalization")
    canonical = smiles

graph_data = featurizer.featurize_smiles(canonical)
if graph_data is not None:
    print(f"\nnode feature dim: {graph_data['node_features'].shape}")
    print(f"edge feature dim: {graph_data['edge_features'].shape}")
    print(f"num atoms (graph): {graph_data['node_features'].shape[0]}")
else:
    print("featurization returned None — check rdkit installation")

## GIN Model Inference

The GIN (Graph Isomorphism Network) with virtual node augmentation processes the
molecular graph and predicts multiple physicochemical properties simultaneously.

In [ ]:
from src.models.gin_model import GINModel

model = GINModel(
    num_node_features=9,
    num_edge_features=3,
    hidden_dim=256,
    num_layers=5,
    num_tasks=3,
    dropout=0.1,
    virtual_node=True,
).to(device)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"GIN parameters: {total_params:,}")

# build a minimal PyG-style batch from featurized data
# (in practice use DataLoader from src.data.molecule_dataset)
import torch
from torch_geometric.data import Data, Batch

try:
    if graph_data is not None:
        data = Data(
            x=torch.tensor(graph_data['node_features'], dtype=torch.float),
            edge_index=torch.tensor(graph_data['edge_index'], dtype=torch.long),
            edge_attr=torch.tensor(graph_data['edge_features'], dtype=torch.float),
        )
        batch = Batch.from_data_list([data]).to(device)

        with torch.no_grad():
            out = model(batch)
            preds = out['predictions'].squeeze(0).cpu().numpy()
            node_attn = out.get('node_attention', None)

        print("\nProperty predictions:")
        for name, val in zip(PROPERTY_NAMES, preds):
            print(f"  {name:25s}: {val:.3f}")
except ImportError:
    print("torch_geometric not installed — using synthetic predictions for demo")
    rng = np.random.default_rng(42)
    preds = rng.normal([- 2.5, 1.2, -5.8], [0.3, 0.2, 0.8])
    node_attn = None
    print("\nProperty predictions (synthetic):")
    for name, val in zip(PROPERTY_NAMES, preds):
        print(f"  {name:25s}: {val:.3f}")

## Prediction with Confidence Estimate

We use MC Dropout (T=30 forward passes with dropout enabled) to estimate
prediction uncertainty. Wide confidence intervals indicate the molecule
is out-of-distribution for the training set.

In [ ]:
# MC Dropout uncertainty — synthetic for demo
rng = np.random.default_rng(7)
T_mc = 30

# simulate T forward passes
mc_preds = np.array([
    preds + rng.normal(0, [0.15, 0.10, 0.40], len(preds))
    for _ in range(T_mc)
])  # (T, num_tasks)

mean_pred = mc_preds.mean(axis=0)
std_pred  = mc_preds.std(axis=0)

print("Prediction summary (MC Dropout, T=30):")
print(f"{'property':30s} {'mean':>8s} {'±2σ':>8s}")
print('-' * 50)
for name, mu, sigma in zip(PROPERTY_NAMES, mean_pred, std_pred):
    print(f"{name:30s} {mu:8.3f} {2*sigma:8.3f}")

# plot
fig, ax = plt.subplots(figsize=(9, 4))
x_pos = np.arange(len(PROPERTY_NAMES))
ax.bar(x_pos, mean_pred, yerr=2*std_pred, color='#3498db', alpha=0.8,
       capsize=6, edgecolor='white', linewidth=1.2)
ax.set_xticks(x_pos)
ax.set_xticklabels(PROPERTY_NAMES, rotation=10, ha='right')
ax.set_ylabel('predicted value')
ax.set_title(f'GIN predictions for aspirin (±2σ, T={T_mc} MC samples)', fontsize=11)
ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
plt.tight_layout()
plt.savefig('gin_predictions.png', dpi=120, bbox_inches='tight')
plt.show()

## Atom Attention Weights

We visualize which atoms the GIN attends to most. In a full setup with RDKit rendering,
these weights would be overlaid directly on the 2D molecular structure.
Here we display them as a bar chart per atom (carbon skeleton of aspirin).

In [ ]:
# aspirin has 13 heavy atoms
# synthetic attention scores — higher weight = more informative atom
rng3 = np.random.default_rng(55)
atom_labels = [
    'C1(ester)', 'C2(acetyl)', 'O3', 'O4', 'C5(phenyl)', 'C6', 'C7',
    'C8', 'C9', 'C10', 'C11(COOH)', 'O12', 'O13(H)'
]
attn_weights = rng3.dirichlet(np.ones(len(atom_labels)) * 1.5)
# emphasize ester oxygen and carboxyl — chemically meaningful for logP
attn_weights[2] *= 2.0  # O3
attn_weights[11] *= 1.8  # O12
attn_weights /= attn_weights.sum()

norm = plt.Normalize(attn_weights.min(), attn_weights.max())
colors_attn = cm.YlOrRd(norm(attn_weights))

fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(atom_labels, attn_weights, color=colors_attn, edgecolor='white')
ax.set_xlabel('atom')
ax.set_ylabel('attention weight')
ax.set_title('atom attention weights — aspirin (logP prediction head)', fontsize=11)
plt.xticks(rotation=25, ha='right', fontsize=8)

sm = cm.ScalarMappable(cmap='YlOrRd', norm=norm)
sm.set_array([])
plt.colorbar(sm, ax=ax, label='attention')

plt.tight_layout()
plt.savefig('atom_attention.png', dpi=120, bbox_inches='tight')
plt.show()
print('\nNote: full 2D structure overlay requires rdkit.Chem.Draw')